In [1]:
import lightgbm
import mlflow
import os
from dotenv import load_dotenv
import sys
import pandas as pd
import numpy as np
import json

In [2]:
pd.set_option("display.max_columns", 100)
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
sys.path.append(src_path)
json_path = os.path.join(data_path, "processed/split_info.json")
with open(json_path) as f:
    json_info = json.load(f)
train_end = json_info.get("train_end")
val_end = json_info.get("validation_end")
from about_data.data_load import load_df
from about_data.split import temporal_split
from features.engineering import create_d_features, create_advanced_time_features, add_distance_features, add_interaction_features
from model.preprocessor_pipe_evalueate import create_pipeline, evaluate_model, get_preprocessor

full_df = load_df(data_path)
train, val, test = temporal_split(full_df, train_end, val_end)

In [3]:
train_base = create_d_features(train)
val_base = create_d_features(val)
map_dfs = {"train": train, "val": val}
map_functions = {"d":create_d_features, "time":create_advanced_time_features, "distance":add_distance_features, "interaction":add_interaction_features}
train_features_dfs = {}
val_features_dfs = {}

train_features_dfs = {"d": train_base}
val_features_dfs = {"d": val_base}

for name, func in map_functions.items():
    train_features_dfs[name] = func(train_base)
    val_features_dfs[name] = func(val_base)

In [4]:
len(train_features_dfs)

4

In [10]:
y_datasets = {}

for name, sample_df in map_dfs.items():
    y_datasets[name] = sample_df['isFraud']

In [7]:
model = lightgbm.LGBMClassifier(n_estimators=300, learning_rate=0.05, objective="binary", metric='average_precision', random_state=42, n_jobs=-1)

In [11]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud-detection-feature-engineering")

results = []

for name, _ in map_functions.items():
    with mlflow.start_run(run_name=f"lightgbm_{name}_features"):
        X_train = train_features_dfs[name]
        X_val = val_features_dfs[name]

        pipe = create_pipeline(model, get_preprocessor(X_train))

        pipe.fit(X_train, y_datasets['train'])

        metrics = evaluate_model(pipe, X_val, y_datasets['val'])

        mlflow.log_param("dataset", name)
        mlflow.log_param("feature_count", X_train.shape[1])
        mlflow.log_param("model", "lightgbm")

        mlflow.log_metrics(metrics)

        results.append(metrics)

[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.080076 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10868
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 4094
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794
🏃 View run lightgbm_d_features at: http://127.0.0.1:5000/#/experiments/4/runs/0c257fc5dc3b4953a80b353150503c33
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.242097 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you

C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['dist_diff' 'dist_sum']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.080218 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11152
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 4097
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794


C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['dist_diff' 'dist_sum']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['dist_diff' 'dist_sum']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


🏃 View run lightgbm_distance_features at: http://127.0.0.1:5000/#/experiments/4/runs/94ec8ab103f1414a88ccbbbc6c3ad7de
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.081312 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13864
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 5453
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794
🏃 View run lightgbm_interaction_features at: http://127.0.0.1:5000/#/experiments/4/runs/6760c468c91c401da6ea0141e334ab3a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
